# JobCompass — Notebook 06: New User Onboarding Service

## What this does
Builds and starts a **FastAPI** application that handles real-time onboarding for both candidates and recruiters.

```
Candidate uploads PDF              Recruiter submits job
        │                                   │
POST /candidate/upload            POST /job/submit
        │                                   │
   task queue (async)              task queue (async)
        │                                   │
run_ingestion_pipeline()        extract_job_fields()
        │                                   │
 quality_gate()                  embed_and_index()
   ≥40 → index                          │
   <40 → cold_start Q&A      match_existing_candidates()
        │                                   │
index_and_notify()              notify_recruiter()
   WebSocket push                  WebSocket push
```

## Prerequisites
- NB01–NB05 complete
- OpenSearch running: `docker start opensearch`
- vLLM OCR server on port 8000 (NB01 model)
- vLLM text server on port 8001 (NB02 model)

## How to run

**Step 1** — Copy the two source files into your `src/` folder:
- `onboarding_workers.py` → `../src/onboarding_workers.py`
- `onboarding_service.py` → `../src/onboarding_service.py`

**Step 2** — Install dependencies (Cell 0)

**Step 3** — Edit the config in Cell 1 to match your paths

**Step 4** — Run all cells top to bottom. The service starts in Cell 3.

**Step 5** — Use the test cells (4–7) to verify everything works.

**To run the service standalone** (outside this notebook, e.g. in production):
```bash
cd src
uvicorn onboarding_service:app --host 0.0.0.0 --port 8080 --workers 1
```

**Interactive API docs** once running: http://127.0.0.1:8080/docs

## API endpoints
| Method | Path | Description |
|--------|------|-------------|
| GET | `/health` | Service health check |
| POST | `/candidate/upload` | Upload resume PDF |
| POST | `/job/submit` | Submit job listing (JSON) |
| GET | `/task/{task_id}` | Poll processing status |
| GET | `/candidate/{id}/matches` | Top job matches for candidate |
| GET | `/job/{id}/matches` | Top candidate matches for job |
| WS | `/ws/{client_id}` | WebSocket for real-time notifications |

In [ ]:
# Cell 0 — Install dependencies
import subprocess, sys

packages = [
    "fastapi>=0.111.0",
    "uvicorn[standard]>=0.29.0",
    "python-multipart",
    "websockets",
    "aiohttp",
    "aiofiles",
    "pdfplumber",
    "pymupdf",
    "orjson",
    "opensearch-py>=2.4.0",
    "sentence-transformers>=3.0.0",
    "transformers>=4.51.0",
    "torch",
    "numpy",
    "requests",
    "httpx",
]
for pkg in packages:
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True, text=True
    )
    print(f"  {'OK' if r.returncode == 0 else 'FAIL'} {pkg}")
print("Done.")

In [ ]:
# Cell 1 — Configuration
# Edit these values to match your setup before running anything else.

import os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# ── Source files ───────────────────────────────────────────────────────────────
# These two files must exist before starting the service.
# Copy them from the notebook output folder into src/.
SRC_DIR = "../src"
os.makedirs(SRC_DIR,           exist_ok=True)
os.makedirs("../data/uploads", exist_ok=True)

# ── Service ────────────────────────────────────────────────────────────────────
SERVICE_HOST = "127.0.0.1"
SERVICE_PORT = 8080

# ── vLLM servers ──────────────────────────────────────────────────────────────
VLLM_OCR_URL    = "http://localhost:8000/v1"   # NB01 OCR model
VLLM_OCR_MODEL  = "/mnt/d/models/qwen-3b"
VLLM_TEXT_URL   = "http://localhost:8001/v1"   # NB02 text extraction model
VLLM_TEXT_MODEL = "/mnt/d/models/qwen-3b"

# ── OpenSearch ─────────────────────────────────────────────────────────────────
OS_HOST          = "localhost"
OS_PORT          = 9200
CANDIDATES_ALIAS = "candidates"
JOBS_ALIAS       = "jobs"

# ── Models ─────────────────────────────────────────────────────────────────────
EMBED_MODEL_NAME    = "Qwen/Qwen3-Embedding-0.6B"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
HF_HOME             = "D:/huggingface_cache"

# ── Onboarding ─────────────────────────────────────────────────────────────────
# Candidates scoring below this get the cold-start questionnaire.
QUALITY_GATE_SCORE = 40

print("Config ready.")
print(f"  Service  : http://{SERVICE_HOST}:{SERVICE_PORT}")
print(f"  OCR      : {VLLM_OCR_URL}  model={VLLM_OCR_MODEL}")
print(f"  Text LLM : {VLLM_TEXT_URL}  model={VLLM_TEXT_MODEL}")
print(f"  OS       : {OS_HOST}:{OS_PORT}")
print(f"  Gate     : quality >= {QUALITY_GATE_SCORE}")

In [ ]:
# Cell 2 — Verify source files and prerequisites

import requests
from pathlib import Path

errors = []

# Check source files exist
for fname in ["onboarding_workers.py", "onboarding_service.py"]:
    p = Path(SRC_DIR) / fname
    if p.exists():
        print(f"  Found : {p}")
    else:
        print(f"  MISSING: {p}")
        errors.append(f"Copy {fname} into {SRC_DIR}/")

# Check OpenSearch
try:
    r = requests.get(f"http://{OS_HOST}:{OS_PORT}", timeout=3)
    print(f"  OpenSearch : OK (HTTP {r.status_code})")
except Exception:
    print(f"  OpenSearch : UNREACHABLE")
    errors.append("Start OpenSearch: docker start opensearch")

# Check vLLM servers (warnings only — service can start without them)
for label, url in [("vLLM OCR (8000)", VLLM_OCR_URL), ("vLLM Text (8001)", VLLM_TEXT_URL)]:
    try:
        r = requests.get(url.replace("/v1", "") + "/health", timeout=3)
        print(f"  {label} : OK")
    except Exception:
        print(f"  {label} : not reachable (start before uploading resumes)")

if errors:
    print()
    print("Fix before continuing:")
    for e in errors:
        print(f"  → {e}")
    raise SystemExit("Prerequisites not met.")
else:
    print()
    print("All prerequisites met ✓")

In [ ]:
# Cell 3 — Set environment variables for the service
# The service reads these at startup via os.getenv()

import os

os.environ["UPLOAD_DIR"]          = "../data/uploads"
os.environ["VLLM_OCR_URL"]        = VLLM_OCR_URL
os.environ["VLLM_OCR_MODEL"]      = VLLM_OCR_MODEL
os.environ["VLLM_TEXT_URL"]       = VLLM_TEXT_URL
os.environ["VLLM_TEXT_MODEL"]     = VLLM_TEXT_MODEL
os.environ["OS_HOST"]             = OS_HOST
os.environ["OS_PORT"]             = str(OS_PORT)
os.environ["CANDIDATES_ALIAS"]    = CANDIDATES_ALIAS
os.environ["JOBS_ALIAS"]          = JOBS_ALIAS
os.environ["EMBED_MODEL_NAME"]    = EMBED_MODEL_NAME
os.environ["RERANKER_MODEL_NAME"] = RERANKER_MODEL_NAME
os.environ["HF_HOME"]             = HF_HOME
os.environ["QUALITY_GATE_SCORE"]  = str(QUALITY_GATE_SCORE)

print("Environment variables set ✓")

In [ ]:
# Cell 4 — Start the service in a background thread
#
# Runs uvicorn inside a daemon thread so this notebook stays interactive.
# Model loading (embedding + reranker) happens on startup and takes ~30-60s.
# Watch the output below for "Service ready" before running test cells.

import sys, os, time, threading, requests

# Add src/ to Python path so the service can import onboarding_workers
src_abs = os.path.abspath(SRC_DIR)
if src_abs not in sys.path:
    sys.path.insert(0, src_abs)

import uvicorn

def _run_server():
    uvicorn.run(
        "onboarding_service:app",
        host      = SERVICE_HOST,
        port      = SERVICE_PORT,
        log_level = "info",
        app_dir   = src_abs,
    )

thread = threading.Thread(target=_run_server, daemon=True)
thread.start()

# Poll until ready (models take a while to load)
print("Waiting for service to start (model loading may take 30-60s)...")
BASE_URL = f"http://{SERVICE_HOST}:{SERVICE_PORT}"

for attempt in range(40):
    time.sleep(3)
    try:
        r = requests.get(f"{BASE_URL}/health", timeout=3)
        if r.status_code == 200:
            h = r.json()
            print()
            print(f"Service ready ✓")
            print(f"  Status       : {h['status']}")
            print(f"  OpenSearch   : {h['opensearch']}")
            print(f"  Models loaded: {h['models_loaded']}")
            print(f"  Device       : {h['device']}")
            print()
            print(f"  API docs     : {BASE_URL}/docs")
            break
    except Exception:
        if (attempt + 1) % 5 == 0:
            print(f"  Still waiting... ({(attempt+1)*3}s)")
else:
    print("Service did not start in time. Check log output above for errors.")

In [ ]:
# Cell 5 — Test: job submission
# Submits a test job and polls until processing completes.

import requests, json, time

test_job = {
    "title"      : "Senior Python Developer",
    "role"       : "Backend Engineer",
    "description": (
        "We are looking for a Senior Python Developer to join our engineering team. "
        "You will design and build scalable backend services using Python, FastAPI, "
        "and PostgreSQL. Experience with Docker, Kubernetes, and AWS is required. "
        "You will work closely with the data engineering team on pipeline architecture "
        "and contribute to system design decisions."
    ),
    "skills_raw" : "Python, FastAPI, PostgreSQL, Docker, Kubernetes, AWS, REST APIs, Git",
    "experience" : "5 to 8 years",
    "company"    : "TechCorp Inc.",
    "location"   : {"city": "Bangalore", "country": "India"},
    "work_type"  : "Full-time",
}

print("Submitting test job...")
r    = requests.post(f"{BASE_URL}/job/submit", json=test_job, params={"client_id": "test-recruiter"})
resp = r.json()
print(f"  HTTP {r.status_code}")
print(f"  task_id : {resp.get('task_id')}")
print(f"  status  : {resp.get('status')}")

job_task_id = resp.get("task_id")
print()
print("Polling for completion...")

for i in range(30):
    time.sleep(4)
    status = requests.get(f"{BASE_URL}/task/{job_task_id}").json()
    print(f"  [{status['status']:12s}] {status.get('message', '')}")
    if status["status"] in ("completed", "failed"):
        if status.get("result"):
            print()
            print("Result:")
            print(json.dumps(status["result"], indent=2))
        break

In [ ]:
# Cell 6 — Test: resume upload
# Picks the first PDF from your dataset and uploads it through the live API.

import requests, json, time
from pathlib import Path

resumes_dir = "../data/raw/resumes"
test_pdf    = None

for pdf in Path(resumes_dir).rglob("*.pdf"):
    test_pdf = str(pdf)
    break

if not test_pdf:
    print("No PDF found in ../data/raw/resumes — skipping.")
    print("You can manually test with:")
    print(f"  curl -X POST {BASE_URL}/candidate/upload -F 'file=@/path/to/resume.pdf'")
else:
    print(f"Uploading: {test_pdf}")

    with open(test_pdf, "rb") as f:
        r = requests.post(
            f"{BASE_URL}/candidate/upload",
            files  = {"file": (Path(test_pdf).name, f, "application/pdf")},
            params = {"client_id": "test-candidate"},
        )

    resp         = r.json()
    cand_task_id = resp.get("task_id")
    print(f"  HTTP {r.status_code}  task_id={cand_task_id}")
    print()
    print("Polling for completion (OCR + extraction takes 30-90s per resume)...")

    for i in range(60):
        time.sleep(5)
        status = requests.get(f"{BASE_URL}/task/{cand_task_id}").json()
        print(f"  [{status['status']:12s}] {status.get('message', '')}")
        if status["status"] in ("completed", "failed", "cold_start"):
            if status.get("result"):
                display = {k: v for k, v in status["result"].items() if k != "content"}
                print()
                print("Result:")
                print(json.dumps(display, indent=2))
            break

In [ ]:
# Cell 7 — Test: get matches via REST endpoints
# Queries matches for an existing candidate and job from your batch dataset.

import requests, orjson
from pathlib import Path

# ── Candidate matches ─────────────────────────────────────────────────────────
cand_id = None
cands_path = "../data/processed/candidates_embedded.jsonl"
if Path(cands_path).exists():
    with open(cands_path, "rb") as f:
        for line in f:
            if line.strip():
                cand_id = orjson.loads(line)["candidate_id"]
                break

if cand_id:
    print(f"Top job matches for candidate: {cand_id}")
    r = requests.get(f"{BASE_URL}/candidate/{cand_id}/matches", params={"top_k": 5})
    if r.status_code == 200:
        for m in r.json()["matches"]:
            print(f"  [{m['score']:.4f}]  {str(m.get('title','?'))[:40]:40s}  {m.get('company','?')}")
    else:
        print(f"  Error: {r.status_code} — {r.text[:200]}")
else:
    print("No candidates_embedded.jsonl found — run NB03 first.")

print()

# ── Job matches ───────────────────────────────────────────────────────────────
job_id = None
jobs_path = "../data/processed/jobs_embedded.jsonl"
if Path(jobs_path).exists():
    with open(jobs_path, "rb") as f:
        for line in f:
            if line.strip():
                job_id = orjson.loads(line)["job_id"]
                break

if job_id:
    print(f"Top candidate matches for job: {job_id}")
    r = requests.get(f"{BASE_URL}/job/{job_id}/matches", params={"top_k": 5})
    if r.status_code == 200:
        for m in r.json()["matches"]:
            print(f"  [{m['score']:.4f}]  {str(m.get('name','?'))[:20]:20s}  {str(m.get('title','?'))[:35]}")
    else:
        print(f"  Error: {r.status_code} — {r.text[:200]}")
else:
    print("No jobs_embedded.jsonl found — run NB03 first.")

In [ ]:
# Cell 8 — Test: WebSocket notification
# Connects a WebSocket client and submits a job to verify real-time push works.

import asyncio, json, threading
import websockets
import requests

WS_URL    = f"ws://{SERVICE_HOST}:{SERVICE_PORT}/ws/ws-test-client"
received  = []
ws_done   = threading.Event()

async def _listen():
    try:
        async with websockets.connect(WS_URL) as ws:
            print("  WebSocket connected ✓")
            for _ in range(20):
                try:
                    msg = await asyncio.wait_for(ws.recv(), timeout=5)
                    data = json.loads(msg)
                    received.append(data)
                    print(f"  WS message: {data}")
                    if data.get("status") in ("completed", "failed"):
                        break
                except asyncio.TimeoutError:
                    break
    except Exception as e:
        print(f"  WebSocket error: {e}")
    finally:
        ws_done.set()

def _run_ws():
    asyncio.run(_listen())

ws_thread = threading.Thread(target=_run_ws, daemon=True)
ws_thread.start()

import time
time.sleep(1)   # give WebSocket time to connect

# Submit a job while WebSocket is listening
test_job_ws = {
    "title"      : "Data Engineer",
    "description": "We need a Data Engineer with Python, Spark, and Airflow experience to build ETL pipelines.",
    "skills_raw" : "Python, Apache Spark, Airflow, SQL, AWS S3",
    "experience" : "3 to 6 years",
    "company"    : "DataFlow Ltd.",
}

print("Submitting job while WebSocket is connected...")
r = requests.post(
    f"{BASE_URL}/job/submit",
    json   = test_job_ws,
    params = {"client_id": "ws-test-client"},
)
print(f"  Submitted: task_id={r.json().get('task_id')}")

ws_done.wait(timeout=120)
print(f"\nTotal WebSocket messages received: {len(received)}")
if received:
    final = received[-1]
    print(f"Final status: {final.get('status')}")
    if final.get('top_candidate_matches'):
        print(f"Top matches: {final['top_candidate_matches'][:3]}")

In [ ]:
# Cell 9 — Summary
print("NB06 complete.")
print()
print("Service is running at:")
print(f"  http://{SERVICE_HOST}:{SERVICE_PORT}")
print(f"  Interactive docs: http://{SERVICE_HOST}:{SERVICE_PORT}/docs")
print()
print("Source files written to:")
print(f"  {SRC_DIR}/onboarding_workers.py")
print(f"  {SRC_DIR}/onboarding_service.py")
print()
print("To run as a standalone service (outside this notebook):")
print(f"  cd {SRC_DIR}")
print(f"  uvicorn onboarding_service:app --host 0.0.0.0 --port 8080 --workers 1")
print()
print("The service will stay alive as long as this notebook kernel is running.")
print("To stop it, shut down the kernel.")